Name: Dengyi Gu

Github Username: DengyiGu

USC ID: 1332838543

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import bootstrap

**1. Time Series Classification Part 1: Feature Creation/Extraction**

**(a)**

In [2]:
data_path = Path("../data/AReM")

The AReM dataset has been downloaded and extracted to the `data/AReM` folder.

**(b)**

In [3]:
train_files = []
test_files = []

for file in sorted(data_path.glob("*/dataset*.csv")):
    activity = file.parent.name
    number = int(file.stem.replace("dataset", ""))

    if activity in ["bending1", "bending2"]:
        test_numbers = [1, 2]
    else:
        test_numbers = [1, 2, 3]

    if number in test_numbers:
        test_files.append(file)
    else:
        train_files.append(file)

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))

Training instances: 69
Test instances: 19


**(c)**

**(i)**

Common time-domain features include minimum, maximum, mean, median, standard deviation, variance, quartiles and autocorrelation.

**(ii)**

In [4]:
all_files = sorted(train_files + test_files)
rows = []

for file in all_files:
    data = pd.read_csv(
        file,
        comment="#",
        header=None,
        sep=r"[,\s]+",
        engine="python",
        usecols=range(7)
    )

    row = {}

    for i, subscript in enumerate(["₁", "₂", "₃", "₄", "₅", "₆"], start=1):
        series = data.iloc[:, i]

        row[f"min{subscript}"] = series.min()
        row[f"max{subscript}"] = series.max()
        row[f"mean{subscript}"] = series.mean()
        row[f"median{subscript}"] = series.median()
        row[f"std{subscript}"] = series.std()
        row[f"1st quart{subscript}"] = series.quantile(0.25)
        row[f"3rd quart{subscript}"] = series.quantile(0.75)

    rows.append(row)

features = pd.DataFrame(rows)
features.index = range(1, len(features) + 1)
features.index.name = "Instance"
features

,min₁,max₁,mean₁,median₁,std₁,1st quart₁,3rd quart₁,min₂,max₂,mean₂,...,std₅,1st quart₅,3rd quart₅,min₆,max₆,mean₆,median₆,std₆,1st quart₆,3rd quart₆
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
2,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
3,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
4,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
5,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,4.223792,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
85,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,4.115750,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
86,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,4.396958,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200


**(iii)**

In [5]:
results = []
rng = np.random.default_rng(42)

for column in features.columns:
    values = features[column].to_numpy()

    result = bootstrap(
        (values,),
        statistic=lambda x: np.std(x, ddof=1),
        vectorized=False,
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        rng=rng
    )

    results.append({
        "Feature": column,
        "Standard deviation": np.std(values, ddof=1),
        "Lower 90% CI": result.confidence_interval.low,
        "Upper 90% CI": result.confidence_interval.high
    })

std_intervals = pd.DataFrame(results).set_index("Feature")
std_intervals

,Standard deviation,Lower 90% CI,Upper 90% CI
Feature,,,
min₁,9.569975,8.259076,10.776174
max₁,4.394362,3.341697,5.293556
mean₁,5.335718,4.699259,5.881845
median₁,5.440054,4.791011,5.991320
std₁,1.772153,1.572605,1.949162
1st quart₁,6.153590,5.581414,6.646648
3rd quart₁,5.138925,4.317729,5.848283
min₂,0.000000,0.000000,0.000000
max₂,5.062729,4.624126,5.401779


**(iv)**

I select mean, maximum, and standard deviation as the three most important time-domain features. The mean describes the overall signal level, the maximum captures the peak value, and the standard deviation measures variability. Together, these features may help distinguish different activities.

**2.ISLR 3.7.4**

**(a)**

The cubic regression is expected to have a lower training RSS. It includes all terms in the linear model plus $X^2$ and $X^3$, allowing it to fit the training data more closely. Even though the true relationship is linear, these additional terms can fit random noise and reduce the training RSS.

**(b)**

The linear regression is expected to have a lower test RSS because the true relationship is linear. The additional $X^2$ and $X^3$ terms in the cubic model may overfit noise in the training data, increasing variance and reducing prediction accuracy on the test data.

**(c)**

The cubic regression is expected to have a lower training RSS. The linear model is a special case of the cubic model, so the additional $X^2$ and $X^3$ terms allow a closer fit to the training data. This holds regardless of how far the true relationship is from linear.

**(d)**

There is not enough information to determine which model will have a lower test RSS. If the true relationship is close to linear, the linear model may perform better because it has lower variance. If the relationship is strongly nonlinear and better captured by the cubic model, the cubic model may perform better because it reduces bias.